## Legacy 02E regression figures (streamlined)

These pre-2025 figures use outputs from `legacy/02E_better-gap-measures.Rmd` and are retained only as a design reference for the new 2025 Paper 1 gap analysis:
- `data/gap_beta_model_data_consistent.csv`
- `data/gap_beta_model_coefficients_core.csv`
- `data/gap_beta_model_coefficients_extension.csv`
- `data/gap_beta_model_coefficients_sensitivity.csv`

These figures emphasize model results and diagnostics rather than the removed descriptive sections.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('darkgrid')
sns.set_palette('colorblind')

FIG_DIR = Path('figs')
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv('data/gap_beta_model_data_consistent.csv')
coef_core = pd.read_csv('data/gap_beta_model_coefficients_core.csv')
coef_ext = pd.read_csv('data/gap_beta_model_coefficients_extension.csv')
coef_sens = pd.read_csv('data/gap_beta_model_coefficients_sensitivity.csv')

print('Model data:', df.shape)
print('Core coef:', coef_core.shape)
print('Extension coef:', coef_ext.shape)
print('Sensitivity coef:', coef_sens.shape)


### 1. Outcome distribution


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['gap_score'].dropna(), bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(df['gap_score'].mean(), linestyle='--', color='darkred', linewidth=1.5, label=f"Mean = {df['gap_score'].mean():.3f}")
ax.set_xlabel('gap_score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Discussion-Level Gap Score')
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / 'gap_score_distribution.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / 'gap_score_distribution.svg', bbox_inches='tight')
plt.show()


### 2. Coefficient plots


In [ ]:
def tidy_coef(df_coef):
    out = df_coef.copy()
    out = out.rename(columns={'Std. Error': 'std_error', 'Pr(>|z|)': 'p_value', 'z value': 'z_value'})
    out['ci_low'] = out['Estimate'] - 1.96 * out['std_error']
    out['ci_high'] = out['Estimate'] + 1.96 * out['std_error']
    out = out[out['term'] != '(Intercept)'].copy()
    return out


def label_term(term):
    mapping = {
        'z_log_hours_since_article_editors_mean': "Editors' picks timing (log,z)",
        'z_num_sentences_editors_mean': "Editors' picks length (sentences,z)",
        'z_log_user_follower_editors_mean': "Editors' picks follower (log,z)",
        'z_log_article_comments': 'Discussion size (log,z)',
        'z_log_votes_pos_mean': 'Mean upvotes per comment (log,z)',
        'z_log_votes_neg_mean': 'Mean downvotes per comment (log,z)',
        'z_log_n_picks': "Number of editors' picks (log,z)",
        'z_follower_diff': 'Follower diff: editors - user top picks (z)',
        'z_controversy': 'Controversy: downvote share (z)'
    }
    if term.startswith('genre1'):
        return term.replace('genre1', 'Genre: ')
    return mapping.get(term, term)


def coef_forest(df_coef, title, fname):
    d = tidy_coef(df_coef)
    d['label'] = d['term'].map(label_term)
    d = d.sort_values('Estimate')

    fig, ax = plt.subplots(figsize=(8, max(4, 0.38 * len(d))))
    ax.errorbar(
        d['Estimate'],
        np.arange(len(d)),
        xerr=[d['Estimate'] - d['ci_low'], d['ci_high'] - d['Estimate']],
        fmt='o',
        color='steelblue',
        ecolor='gray',
        capsize=2
    )
    ax.axvline(0, color='darkred', linestyle='--', linewidth=1)
    ax.set_yticks(np.arange(len(d)))
    ax.set_yticklabels(d['label'])
    ax.set_xlabel('Coefficient estimate (logit scale)')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{fname}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{fname}.svg', bbox_inches='tight')
    plt.show()


coef_forest(coef_core, 'Core model coefficients (02C-consistent)', 'gap_beta_core_coefficients')
coef_forest(coef_ext, 'Extension model coefficients', 'gap_beta_extension_coefficients')
coef_forest(coef_sens, 'Sensitivity model coefficients (no n_picks)', 'gap_beta_sensitivity_coefficients')


### 3. Cross-model comparison (shared terms)


In [ ]:
core_t = tidy_coef(coef_core)[['term', 'Estimate', 'std_error']].assign(model='core')
ext_t = tidy_coef(coef_ext)[['term', 'Estimate', 'std_error']].assign(model='extension')
sens_t = tidy_coef(coef_sens)[['term', 'Estimate', 'std_error']].assign(model='sensitivity')

all_t = pd.concat([core_t, ext_t, sens_t], ignore_index=True)
shared_terms = all_t.groupby('term')['model'].nunique()
shared_terms = shared_terms[shared_terms == 3].index
plot_t = all_t[all_t['term'].isin(shared_terms)].copy()
plot_t['label'] = plot_t['term'].map(label_term)

order = (
    plot_t.groupby('label')['Estimate']
    .mean()
    .sort_values()
    .index
)
plot_t['label'] = pd.Categorical(plot_t['label'], categories=order, ordered=True)
plot_t = plot_t.sort_values('label')

fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(order))))
for model, marker, color in [('core', 'o', '#1f77b4'), ('extension', 's', '#ff7f0e'), ('sensitivity', '^', '#2ca02c')]:
    d = plot_t[plot_t['model'] == model]
    ax.errorbar(
        d['Estimate'],
        d['label'],
        xerr=1.96 * d['std_error'],
        fmt=marker,
        color=color,
        ecolor=color,
        capsize=2,
        linestyle='none',
        label=model
    )

ax.axvline(0, color='darkred', linestyle='--', linewidth=1)
ax.set_xlabel('Coefficient estimate (logit scale)')
ax.set_title('Shared-term coefficient comparison across models')
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / 'gap_beta_shared_term_comparison.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / 'gap_beta_shared_term_comparison.svg', bbox_inches='tight')
plt.show()


### 4. Collinearity diagnostic heatmap (extension numeric block)


In [ ]:
ext_cols = [
    'z_log_hours_since_article_editors_mean',
    'z_num_sentences_editors_mean',
    'z_log_user_follower_editors_mean',
    'z_log_article_comments',
    'z_log_votes_pos_mean',
    'z_log_votes_neg_mean',
    'z_log_n_picks',
    'z_follower_diff',
    'z_controversy'
]

corr = df[ext_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=False, ax=ax)
ax.set_title('Extension predictor correlation matrix')
fig.tight_layout()
fig.savefig(FIG_DIR / 'gap_beta_extension_correlation_heatmap.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / 'gap_beta_extension_correlation_heatmap.svg', bbox_inches='tight')
plt.show()


### 5. Key predictor vs outcome (raw associations)


In [ ]:
plot_specs = [
    ('z_log_hours_since_article_editors_mean', "Editors' picks timing (log,z)"),
    ('z_num_sentences_editors_mean', "Editors' picks length (sentences,z)"),
    ('z_log_user_follower_editors_mean', "Editors' picks follower (log,z)"),
    ('z_follower_diff', 'Follower diff (z)')
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey=True)
axes = axes.flatten()

for ax, (xcol, xlabel) in zip(axes, plot_specs):
    d = df[[xcol, 'gap_score']].dropna()
    ax.scatter(d[xcol], d['gap_score'], alpha=0.25, s=10, color='steelblue')
    if len(d) >= 5:
        z = np.polyfit(d[xcol], d['gap_score'], 1)
        xline = np.linspace(d[xcol].min(), d[xcol].max(), 100)
        ax.plot(xline, np.polyval(z, xline), color='darkred', linewidth=1)
    ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('gap_score')

fig.suptitle('Raw bivariate associations (descriptive only)', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'gap_beta_key_raw_associations.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / 'gap_beta_key_raw_associations.svg', bbox_inches='tight')
plt.show()
